<a href="https://colab.research.google.com/github/AAYU3027/LEARING-PROJECTS/blob/main/MINOR_PROJECT_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import pandas as pd
import numpy as np

# ==========================================
# SETUP & CONFIGURATION
# ==========================================
FILE_PATH = 'rahul_transactions.csv'

# Vendor Dictionary Mapping (Expand this based on your EDA)
VENDOR_DICT = {
    'Swiggy': ['SWIGGY', 'BUNDL'],
    'Zomato': ['ZOMATO'],
    'Zepto': ['ZEPTO', 'BLINKIT', 'INSTAMART'],
    'Amazon': ['AMAZON', 'AMZN'],
    'Myntra': ['MYNTRA'],
    'Zerodha': ['ZERODHA', 'COIN'],
    'BookMyShow': ['BOOKMYSHOW', 'BMS'],
    'Netflix': ['NETFLIX'],
    'Spotify': ['SPOTIFY'],
    'Uber': ['UBER'],
    'Ola': ['OLA'],
    'Starbucks': ['STARBUCKS', 'TATA STARBUCKS'],
    'Third Wave': ['THIRD WAVE'],
    'Bescom': ['BESCOM'],
    'Jio': ['JIO'],
    'P2P Transfer': ['UPI-PRIYA', 'UPI-ANKIT', 'UPI-RAHUL'],
    'Cash Withdrawal': ['ATM-WDL']
}

# Category Dictionary Mapping
CATEGORY_DICT = {
    'Swiggy': 'Food Delivery',
    'Zomato': 'Food Delivery',
    'Zepto': 'Quick Commerce',
    'Amazon': 'E-commerce',
    'Myntra': 'E-commerce',
    'Zerodha': 'Investments',
    'BookMyShow': 'Entertainment',
    'Netflix': 'Subscriptions',
    'Spotify': 'Subscriptions',
    'Uber': 'Transport',
    'Ola': 'Transport',
    'Starbucks': 'Cafe',
    'Third Wave': 'Cafe',
    'Bescom': 'Utilities',
    'Jio': 'Utilities',
    'P2P Transfer': 'Personal Transfer',
    'Cash Withdrawal': 'Cash Withdrawal'
}

# ==========================================
# FEATURE 1: THE TRANSACTION PARSER
# ==========================================
def parse_transactions(file_path):
    df = pd.read_csv(file_path)
    initial_rows = len(df)

    # 1. Parse Dates (handles 4 formats automatically)
    df['date'] = pd.to_datetime(df['Date'], errors='coerce', dayfirst=True)

    # 2. Parse Amounts (handles 3 formats)
    # Strip symbols, commas, and spaces
    amount_str = df['Amount'].astype(str).str.replace('$', '', regex=False)\
                                         .str.replace('Rs.', '', regex=False)\
                                         .str.replace(',', '', regex=False)\
                                         .str.strip()
    df['amount'] = pd.to_numeric(amount_str, errors='coerce')

    # 3. Standardize Type (Debit/Credit)
    df['type'] = df['Type'].astype(str).str.lower().str.strip()
    df['type'] = df['type'].apply(lambda x: 'debit' if x == 'dr' or x == 'debit' else 'credit')

    # 4. Drop Duplicates
    df = df.drop_duplicates()
    final_rows = len(df)

    print(f"Parsed {final_rows} transactions. Dropped {initial_rows - final_rows} duplicates.")
    return df

# ==========================================
# FEATURE 2: VENDOR EXTRACTOR
# ==========================================
def extract_vendor(desc):
    desc_upper = str(desc).upper()
    for canonical_name, keywords in VENDOR_DICT.items():
        for kw in keywords:
            if kw.upper() in desc_upper:
                return canonical_name
    return 'Uncategorised'

# ==========================================
# FEATURE 3-8: ANALYSIS & REPORT GENERATION
# ==========================================
def run_spend_dna():
    # Load and clean data
    try:
        df = parse_transactions(FILE_PATH)
    except FileNotFoundError:
        print(f"Error: {FILE_PATH} not found. Please ensure it is in the same directory.")
        return

    # F2: Apply Vendor Extractor
    df['vendor_clean'] = df['Description'].apply(extract_vendor)

    # F3: Apply Category Tagger
    df['category'] = df['vendor_clean'].map(CATEGORY_DICT).fillna('Uncategorised')

    # Time Data Extraction
    df['hour'] = df['Time'].astype(str).str[:2].astype(int)
    df['month'] = df['date'].dt.month
    df['month_name'] = df['date'].dt.month_name().str[:3]

    # Filter Debits vs Credits for spending analysis
    debits = df[df['type'] == 'debit'].copy()
    credits = df[df['type'] == 'credit'].copy()

    # F4: Spending Overview Calculations
    total_credits = credits['amount'].sum()
    total_debits = debits['amount'].sum()
    net_change = total_credits - total_debits
    savings_rate = (net_change / total_credits) * 100 if total_credits > 0 else 0

    # F7: Anomaly Detection (Z-Score)
    debits['category_mean'] = debits.groupby('category')['amount'].transform('mean')
    debits['category_std'] = debits.groupby('category')['amount'].transform('std')
    debits['z_score'] = (debits['amount'] - debits['category_mean']) / debits['category_std']
    anomalies = debits[debits['z_score'] > 2].sort_values('z_score', ascending=False)

    # ==========================================
    # PRINTING THE ASCII REPORT
    # ==========================================
    print("\n" + "="*66)
    print("SpendDNA REPORT")
    print("RAHUL SHARMA")
    print(f"Jan to Jun 2024 | 6 months | {len(df)} transactions")
    print("="*66)

    print("\nEXECUTIVE SUMMARY")
    print(f"Total credits    : Rs. {total_credits:,.0f}")
    print(f"Total debits     : Rs. {total_debits:,.0f}")
    print(f"Net change       : Rs. {net_change:,.0f} {'(overspending)' if net_change < 0 else '(saving)'}")
    print(f"Savings rate     : {savings_rate:.1f}% {'(BURNING SAVINGS)' if savings_rate < 0 else ''}")
    print(f"Transactions     : {len(df)}")
    print(f"Unique vendors   : {df['vendor_clean'].nunique()}")

    print("\nTOP CATEGORIES (% of debit total)")
    cat_totals = debits[~debits['category'].isin(['Personal Transfer', 'Cash Withdrawal'])].groupby('category')['amount'].sum().sort_values(ascending=False)
    for cat, amt in cat_totals.head(5).items():
        pct = (amt / total_debits) * 100
        bars = "#" * int(pct)
        print(f"{cat:<16} {bars:<18} {pct:>4.1f}%   Rs. {amt:,.0f}")

    print("\nTOP VENDORS")
    vendor_totals = debits[~debits['category'].isin(['Personal Transfer', 'Cash Withdrawal'])].groupby('vendor_clean').agg(total=('amount', 'sum'), count=('amount', 'count')).sort_values('total', ascending=False)
    for vendor, row in vendor_totals.head(5).iterrows():
        print(f"{vendor:<16} Rs. {row['total']:>8,.0f}   ({row['count']:>3.0f} orders)")

    # F6: Time-Of-Day Patterns
    print("\nTIME-OF-DAY PATTERNS")
    food_delivery = debits[debits['category'] == 'Food Delivery']
    if not food_delivery.empty:
        peak_food_hour = food_delivery['hour'].mode().iloc[0]
        late_night_food = food_delivery[(food_delivery['hour'] >= 21) | (food_delivery['hour'] <= 2)]
        late_night_pct = (len(late_night_food) / len(food_delivery)) * 100
        print(f"Food Delivery peaks: {peak_food_hour:02d}:00 ({late_night_pct:.0f}% of orders late night)")

    cafe_spend = debits[debits['category'] == 'Cafe']
    if not cafe_spend.empty:
        cafe_peaks = cafe_spend['hour'].value_counts().head(2).index.tolist()
        print(f"Cafe peaks:          {cafe_peaks[0]:02d}:00 & {cafe_peaks[1]:02d}:00 (morning runs)")

    # F5: Monthly Trend Analysis (Food Delivery)
    print("\nMONTHLY TREND (Food Delivery)")
    if not food_delivery.empty:
        monthly_food = food_delivery.groupby('month_name', sort=False)['amount'].sum()
        # Sort by actual month order
        months_order = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun']
        for m in months_order:
            if m in monthly_food:
                amt = monthly_food[m]
                bars = "#" * int(amt / 3000) # Scaled for visibility
                print(f"{m}   Rs. {amt:>6,.0f} {bars}")

    print("\nTOP ANOMALIES (3+ stddev from category mean)")
    for _, row in anomalies.head(3).iterrows():
        # Check if the date is not NaT before formatting
        if pd.notna(row['date']):
            date_str = row['date'].strftime('%d %b')
        else:
            date_str = 'N/A'
        print(f"{date_str:<6} {row['vendor_clean']:<14} Rs. {row['amount']:>6,.0f}  (z={row['z_score']:.1f})")

    # F8: Archetype Detection
    print("\nRAHUL'S SPENDING ARCHETYPES")
    food_pct = (cat_totals.get('Food Delivery', 0) + cat_totals.get('Restaurants', 0) + cat_totals.get('Cafe', 0)) / total_debits * 100
    qcom_pct = cat_totals.get('Quick Commerce', 0) / total_debits * 100
    ecom_pct = cat_totals.get('E-commerce', 0) / total_debits * 100
    inv_pct = cat_totals.get('Investments', 0) / total_debits * 100
    transport_pct = cat_totals.get('Transport', 0) / total_debits * 100
    subs_count = debits[debits['category'] == 'Subscriptions']['vendor_clean'].nunique()

    if food_pct > 25:
        print(f"-> THE FOODIE               ({food_pct:.1f}% on food)")
    if qcom_pct > 15:
        print(f"-> THE QUICK COMMERCE       ({qcom_pct:.1f}% on Q-com)")
    if ecom_pct > 15:
        print(f"-> THE SHOPAHOLIC           ({ecom_pct:.1f}% on e-commerce)")
    if inv_pct > 15:
        print(f"-> THE INVESTOR             ({inv_pct:.1f}% on SIPs)")
    if not food_delivery.empty and late_night_pct > 50:
        print(f"-> THE LATE-NIGHT SNACKER   ({late_night_pct:.0f}% food after 9 PM)")
    if transport_pct > 10:
        print(f"-> THE CAB COMMUTER         ({transport_pct:.1f}% on transport)")
    if subs_count >= 5:
        print(f"-> THE SUBSCRIPTION LOVER   ({subs_count} active subs)")
    if savings_rate < 10:
        print(f"-> THE YOLO SPENDER         (savings rate {savings_rate:.0f}%)")
    elif savings_rate > 40:
        print(f"-> THE DISCIPLINED SAVER    (savings rate {savings_rate:.0f}%)")

    print("="*66)

# Execute the project
if __name__ == "__main__":
    run_spend_dna()

Parsed 1310 transactions. Dropped 18 duplicates.

SpendDNA REPORT
RAHUL SHARMA
Jan to Jun 2024 | 6 months | 1310 transactions

EXECUTIVE SUMMARY
Total credits    : Rs. 254,818
Total debits     : Rs. 1,236,193
Net change       : Rs. -981,375 (overspending)
Savings rate     : -385.1% (BURNING SAVINGS)
Transactions     : 1310
Unique vendors   : 18

TOP CATEGORIES (% of debit total)
Uncategorised    ########################################## 42.1%   Rs. 520,594
E-commerce       ####################### 23.6%   Rs. 292,190
Investments      ##############     14.6%   Rs. 180,000
Food Delivery    ########            8.3%   Rs. 102,135
Quick Commerce   ###                 3.5%   Rs. 43,434

TOP VENDORS
Uncategorised    Rs.  520,594   (325 orders)
Amazon           Rs.  240,824   ( 62 orders)
Zerodha          Rs.  180,000   ( 12 orders)
Swiggy           Rs.   65,448   (153 orders)
Myntra           Rs.   51,366   ( 12 orders)

TIME-OF-DAY PATTERNS
Food Delivery peaks: 20:00 (21% of orders late nig